# Lab 3 — Avaliação de Agentes RAG com RAGAS e LangFuse
## MBA em RAG & CAG Aplicados a Direito e Segurança Pública | Aula 10

**Objetivo:** Avaliar agentes RAG jurídicos usando:
- **RAGAS:** Faithfulness, Answer Relevancy (qualidade da resposta)
- **Métricas de trajetória:** Tool Recall, Tool Precision, trajetória mínima
- **LangFuse:** Observabilidade, análise de traces e custo estimado

**Duração estimada:** 90 minutos

---

### Por que avaliar agentes é diferente de avaliar RAG simples?

Em RAG simples: avaliamos `pergunta → contexto → resposta`.

Em agentes RAG: avaliamos `pergunta → trajetória de tools → resposta`.

Um agente pode dar uma **resposta certa pelo caminho errado** (usando mais tools que o necessário) — isso aumenta custo e latência desnecessariamente.

In [ ]:
!pip install -q ragas deepeval langfuse langchain langchain-openai datasets

import os
import sqlite3
import time
import json
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from datasets import Dataset

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get('LANGFUSE_PUBLIC_KEY')
    os.environ["LANGFUSE_SECRET_KEY"] = userdata.get('LANGFUSE_SECRET_KEY')
except:
    pass

print("✅ Dependências instaladas")

In [ ]:
# PASSO 1: Dataset de avaliação — ground truth jurídico
# Cada entrada tem: pergunta, resposta ground truth, contextos corretos e trajetória esperada

DATASET_AVALIACAO = [
    {
        "id": "caso_001",
        "question": "Qual é o resultado do HC 123456 julgado pelo STF?",
        "ground_truth": (
            "O HC 123456 foi julgado pelo Min. Alexandre de Moraes em 15/03/2023. "
            "A ordem foi concedida. O caso tratava de prisão preventiva em crime de "
            "corrupção passiva, onde o STF entendeu que faltava fundamentação concreta."
        ),
        "contextos_corretos": [
            "HC 123456 (STF, Min. Alexandre de Moraes, 2023-03-15): "
            "Prisão preventiva em crime de corrupção passiva. "
            "Necessidade de fundamentação concreta. Resultado: Ordem concedida."
        ],
        "trajetoria_esperada": ["search_docs"],  # apenas 1 ferramenta
        "complexidade": "single_step",
        "custo_benchmark_usd": 0.0005
    },
    {
        "id": "caso_002",
        "question": "O que é crime de peculato e como o STF tem julgado casos recentes?",
        "ground_truth": (
            "Peculato é crime contra a administração pública (Art. 312 CP), onde funcionário "
            "público se apropria de bem público. Em 2023, o STF condenou parlamentar federal "
            "no AP 987, impondo 12 anos de pena e perda do mandato."
        ),
        "contextos_corretos": [
            "AP 987 (STF, 2023-11-05): Crime de peculato. Condenação com perda de mandato.",
            "Código Penal, Art. 312: Peculato — apropriar-se o funcionário público de dinheiro, "
            "valor ou qualquer outro bem móvel, público ou particular."
        ],
        "trajetoria_esperada": ["search_docs", "query_db"],
        "complexidade": "multi_step",
        "custo_benchmark_usd": 0.002
    },
    {
        "id": "caso_003",
        "question": "Quais organizações criminosas foram investigadas em 2024 e qual a lei aplicável?",
        "ground_truth": (
            "Em 2024, foi registrado BO-2024-012 em Curitiba/PR: grupo com 12 membros "
            "operando lavagem via imobiliárias. A legislação aplicável é a Lei 12.850/2013, "
            "que define ORCRIM e prevê colaboração premiada e infiltração de agentes."
        ),
        "contextos_corretos": [
            "BO-2024-012: Organização criminosa com 12 membros. Lavagem via imobiliárias. Curitiba/PR.",
            "Lei 12.850/2013: Define organização criminosa. Art. 4 - Colaboração premiada."
        ],
        "trajetoria_esperada": ["query_db", "query_db"],
        "complexidade": "multi_step",
        "custo_benchmark_usd": 0.002
    }
]

print(f"✅ Dataset de avaliação carregado: {len(DATASET_AVALIACAO)} casos")
for caso in DATASET_AVALIACAO:
    print(f"  📋 {caso['id']}: {caso['question'][:60]}...")
    print(f"     Complexidade: {caso['complexidade']} | Tools esperadas: {caso['trajetoria_esperada']}")

In [ ]:
# PASSO 2: Métricas de Trajetória
# Estas métricas avaliam se o AGENTE usou as ferramentas certas

def calcular_metricas_trajetoria(trajetoria_real: list, trajetoria_esperada: list) -> dict:
    """
    Avalia a eficiência da trajetória do agente.
    
    Métricas:
    - tool_recall: cobertura das ferramentas necessárias
    - tool_precision: precisão (sem ferramentas desnecessárias)
    - tool_f1: média harmônica de recall e precision  
    - trajetoria_minima: True se usou exatamente o necessário
    - overhead_pct: % de tool calls extras além do mínimo
    """
    esperado_set = set(trajetoria_esperada)
    real_set = set(trajetoria_real)
    real_count = len(trajetoria_real)
    esperado_count = len(trajetoria_esperada)
    
    if not real_set:
        return {"tool_recall": 0.0, "tool_precision": 1.0, "tool_f1": 0.0,
                "trajetoria_minima": (esperado_count == 0), "overhead_pct": 0.0}
    
    intersecao = esperado_set & real_set
    recall = len(intersecao) / len(esperado_set) if esperado_set else 1.0
    precision = len(intersecao) / len(real_set) if real_set else 1.0
    f1 = 2 * recall * precision / (recall + precision) if (recall + precision) > 0 else 0.0
    
    minima = (real_set == esperado_set) and (real_count == esperado_count)
    overhead = max(0, real_count - esperado_count) / esperado_count if esperado_count > 0 else 0.0
    
    return {
        "tool_recall": round(recall, 3),
        "tool_precision": round(precision, 3),
        "tool_f1": round(f1, 3),
        "trajetoria_minima": minima,
        "overhead_pct": round(overhead * 100, 1)
    }


# Teste das métricas com cenários ilustrativos
print("📊 EXEMPLOS DE MÉTRICAS DE TRAJETÓRIA\n")
print(f"{'Cenário':<30} {'Real':<30} {'Esperado':<20} {'Recall':>7} {'Prec':>6} {'F1':>6} {'Mínima':>8}")
print("-" * 105)

cenarios_teste = [
    ("Perfeito", ["search_docs"], ["search_docs"]),
    ("Incompleto", ["web_search"], ["search_docs", "query_db"]),
    ("Redundante (OK)", ["search_docs", "query_db", "search_docs"], ["search_docs", "query_db"]),
    ("Caminho errado", ["web_search", "query_db"], ["search_docs", "query_db"]),
    ("Multi-step correto", ["query_db", "search_docs"], ["query_db", "search_docs"]),
]

for nome, real, esperado in cenarios_teste:
    m = calcular_metricas_trajetoria(real, esperado)
    icon = "✅" if m["trajetoria_minima"] else ("⚠️ " if m["tool_recall"] > 0.5 else "❌")
    print(f"{nome:<30} {str(real):<30} {str(esperado):<20} "
          f"{m['tool_recall']:>7.0%} {m['tool_precision']:>6.0%} {m['tool_f1']:>6.0%} "
          f"{icon}{str(m['trajetoria_minima']):>6}")

In [ ]:
# PASSO 3: Avaliação com RAGAS — Qualidade da Resposta

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy

# Dataset no formato RAGAS
ragas_data = {
    "question": [caso["question"] for caso in DATASET_AVALIACAO],
    "answer": [
        # Respostas simuladas do agente (em produção, geradas pelo agente)
        "O HC 123456 foi julgado em 15/03/2023 pelo STF. O relator foi o Min. Alexandre de Moraes "
        "e a ordem foi concedida. O tribunal determinou que a prisão preventiva carecia de "
        "fundamentação concreta e individual.",
        
        "Peculato é o crime do Art. 312 do Código Penal, onde o funcionário público se apropria "
        "de bem público. Em novembro de 2023, o STF condenou um parlamentar no AP 987 a pena "
        "privativa de liberdade com decretação de perda do mandato.",
        
        "Em 2024, destaca-se o BO-2024-012 de Curitiba onde 12 integrantes de organização "
        "criminosa foram presos por lavagem de dinheiro via imobiliárias. A lei aplicável é "
        "a Lei 12.850/2013 (Lei das Organizações Criminosas)."
    ],
    "contexts": [caso["contextos_corretos"] for caso in DATASET_AVALIACAO],
    "ground_truth": [caso["ground_truth"] for caso in DATASET_AVALIACAO]
}

ragas_dataset = Dataset.from_dict(ragas_data)

print("📊 Dataset RAGAS preparado:")
print(f"   {len(ragas_data['question'])} casos de avaliação")
print(f"   Métricas: Faithfulness, Answer Relevancy")

if os.getenv("OPENAI_API_KEY"):
    print("\n🔍 Executando avaliação RAGAS...")
    try:
        resultado_ragas = evaluate(
            ragas_dataset,
            metrics=[faithfulness, answer_relevancy]
        )
        
        df_ragas = resultado_ragas.to_pandas()
        
        print("\n📊 RESULTADOS RAGAS:")
        print(f"  Faithfulness (fidelidade):  {resultado_ragas['faithfulness']:.3f}")
        print(f"  Answer Relevancy:           {resultado_ragas['answer_relevancy']:.3f}")
        print("\n  Interpretação:")
        print("  - Faithfulness > 0.9: resposta fiel ao contexto ✅")
        print("  - Answer Relevancy > 0.8: resposta relevante para a pergunta ✅")
        print("\nDetalhamento:")
        print(df_ragas[["question", "faithfulness", "answer_relevancy"]].to_string(index=False))
    except Exception as e:
        print(f"Erro: {e}")
        print("(Verifique que ragas e openai estão instalados corretamente)")
else:
    print("\n📊 RESULTADOS ESPERADOS (referência para sistema bem calibrado):")
    print("  Faithfulness:      0.94 — resposta praticamente fiel ao contexto")
    print("  Answer Relevancy:  0.87 — resposta relevante para a pergunta")
    
    # Dados simulados para demonstração
    df_ragas = pd.DataFrame([
        {"question": caso["question"][:50], "faithfulness": f, "answer_relevancy": r}
        for caso, f, r in zip(DATASET_AVALIACAO, [0.95, 0.93, 0.94], [0.89, 0.86, 0.87])
    ])
    print("\n" + df_ragas.to_string(index=False))

In [ ]:
# PASSO 4: Avaliação Completa — Trajetória + Qualidade

# Trajetórias simuladas do agente (em produção, registradas durante execução)
trajetorias_reais_simuladas = [
    ["search_docs"],                          # caso_001: perfeito
    ["search_docs", "query_db", "web_search"], # caso_002: extra web_search
    ["query_db", "query_db"],                  # caso_003: perfeito
]

print("📊 AVALIAÇÃO COMPLETA DO AGENTE RAG\n")
print("=" * 80)

resultados_completos = []
for i, caso in enumerate(DATASET_AVALIACAO):
    traj_real = trajetorias_reais_simuladas[i]
    traj_esperada = caso["trajetoria_esperada"]
    metricas_traj = calcular_metricas_trajetoria(traj_real, traj_esperada)
    
    # Calcula custo estimado
    custo_real = len(traj_real) * 0.0005  # estimativa por tool call
    custo_benchmark = caso["custo_benchmark_usd"]
    overhead_custo = (custo_real / custo_benchmark - 1) * 100 if custo_benchmark > 0 else 0
    
    # Notas RAGAS simuladas (em produção, vêm do RAGAS)
    faithfulness_sim = [0.95, 0.93, 0.94][i]
    relevancy_sim = [0.89, 0.86, 0.87][i]
    
    resultado = {
        "id": caso["id"],
        "complexidade": caso["complexidade"],
        "tools_esperadas": traj_esperada,
        "tools_usadas": traj_real,
        "tool_recall": metricas_traj["tool_recall"],
        "tool_precision": metricas_traj["tool_precision"],
        "tool_f1": metricas_traj["tool_f1"],
        "trajetoria_minima": metricas_traj["trajetoria_minima"],
        "overhead_pct": metricas_traj["overhead_pct"],
        "faithfulness": faithfulness_sim,
        "answer_relevancy": relevancy_sim,
        "custo_real_usd": custo_real,
        "overhead_custo_pct": overhead_custo
    }
    resultados_completos.append(resultado)
    
    print(f"\n{'='*80}")
    print(f"📋 {caso['id']} | Complexidade: {caso['complexidade']}")
    print(f"   Tools esperadas: {traj_esperada}")
    print(f"   Tools usadas:    {traj_real}")
    print(f"   \n   📊 Métricas de Trajetória:")
    print(f"      Recall: {metricas_traj['tool_recall']:.0%} | Precision: {metricas_traj['tool_precision']:.0%} | F1: {metricas_traj['tool_f1']:.0%}")
    print(f"      Trajetória mínima: {'✅ Sim' if metricas_traj['trajetoria_minima'] else '⚠️  Não'} | Overhead: {metricas_traj['overhead_pct']:.0f}%")
    print(f"   \n   📊 Métricas de Qualidade (RAGAS):")
    print(f"      Faithfulness: {faithfulness_sim:.3f} | Answer Relevancy: {relevancy_sim:.3f}")
    print(f"   \n   💰 Custo: ${custo_real:.4f} (overhead: {overhead_custo:.0f}%)")

df_completo = pd.DataFrame(resultados_completos)

print(f"\n{'='*80}")
print("📊 RESUMO GERAL")
print(f"{'='*80}")
print(f"  Tool F1 médio:          {df_completo['tool_f1'].mean():.3f}")
print(f"  Faithfulness médio:     {df_completo['faithfulness'].mean():.3f}")
print(f"  Answer Relevancy médio: {df_completo['answer_relevancy'].mean():.3f}")
print(f"  Overhead de custo:      {df_completo['overhead_custo_pct'].mean():.0f}% acima do benchmark")

In [ ]:
# PASSO 5: Dashboard de Avaliação

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Dashboard de Avaliação — Agente RAG Jurídico", fontsize=14, fontweight="bold")

casos = [r["id"] for r in resultados_completos]
cores_qualidade = ["#2ecc71" if v > 0.9 else "#f39c12" if v > 0.8 else "#e74c3c"
                   for v in df_completo["faithfulness"]]

# Gráfico 1: Faithfulness e Answer Relevancy
x = range(len(casos))
width = 0.35
axes[0,0].bar([i - width/2 for i in x], df_completo["faithfulness"], width,
              label="Faithfulness", color="#3498db", alpha=0.8)
axes[0,0].bar([i + width/2 for i in x], df_completo["answer_relevancy"], width,
              label="Answer Relevancy", color="#2ecc71", alpha=0.8)
axes[0,0].axhline(y=0.9, color="red", linestyle="--", alpha=0.5, label="Threshold (0.9)")
axes[0,0].set_xticks(x)
axes[0,0].set_xticklabels(casos, rotation=15)
axes[0,0].set_ylim(0, 1.1)
axes[0,0].set_title("Métricas RAGAS por Caso")
axes[0,0].legend()

# Gráfico 2: Tool F1 por caso
cores_traj = ["#2ecc71" if v > 0.95 else "#f39c12" if v > 0.7 else "#e74c3c"
              for v in df_completo["tool_f1"]]
axes[0,1].bar(casos, df_completo["tool_f1"], color=cores_traj, alpha=0.8)
axes[0,1].axhline(y=1.0, color="green", linestyle="--", alpha=0.5, label="Ideal")
axes[0,1].set_ylim(0, 1.1)
axes[0,1].set_title("Tool F1 por Caso (Eficiência de Trajetória)")
axes[0,1].tick_params(axis='x', rotation=15)
axes[0,1].legend()

# Gráfico 3: Overhead de custo
cores_custo = ["#2ecc71" if v == 0 else "#f39c12" if v < 50 else "#e74c3c"
               for v in df_completo["overhead_custo_pct"]]
axes[1,0].bar(casos, df_completo["overhead_custo_pct"], color=cores_custo, alpha=0.8)
axes[1,0].set_title("Overhead de Custo (%)")
axes[1,0].set_ylabel("%")
axes[1,0].tick_params(axis='x', rotation=15)

# Gráfico 4: Radar — Visão Geral
from matplotlib.patches import FancyArrowPatch
metricas_media = {
    "Tool Recall": df_completo["tool_recall"].mean(),
    "Tool Precision": df_completo["tool_precision"].mean(),
    "Faithfulness": df_completo["faithfulness"].mean(),
    "Relevancy": df_completo["answer_relevancy"].mean(),
    "Eficiência\n(1 - overhead/100)": max(0, 1 - df_completo["overhead_custo_pct"].mean() / 100)
}
nomes = list(metricas_media.keys())
valores = list(metricas_media.values())
axes[1,1].barh(nomes, valores, color=["#3498db", "#3498db", "#2ecc71", "#2ecc71", "#f39c12"], alpha=0.8)
axes[1,1].axvline(x=0.9, color="red", linestyle="--", alpha=0.5, label="Threshold")
axes[1,1].set_xlim(0, 1.1)
axes[1,1].set_title("Visão Geral — Médias")
for i, v in enumerate(valores):
    axes[1,1].text(v + 0.01, i, f"{v:.2f}", va="center")

plt.tight_layout()
plt.savefig("avaliacao_agente_rag.png", dpi=100, bbox_inches="tight")
plt.show()
print("✅ Dashboard salvo em: avaliacao_agente_rag.png")

In [ ]:
# PASSO 6: Configuração e uso do LangFuse
# Tutorial de como interpretar traces de agentes

print("🔍 GUIA PRÁTICO: INTERPRETANDO TRACES NO LANGFUSE")
print("=" * 60)

# Simula um trace que seria enviado ao LangFuse
trace_simulado = {
    "trace_id": "trace-001-juridico",
    "session_id": "aula10-lab3",
    "input": "Qual a situação jurídica de lavagem de dinheiro via cripto?",
    "output": "[resposta do agente aqui]",
    "total_tokens": 4823,
    "custo_estimado_usd": 0.00144,
    "latencia_total_ms": 12400,
    "spans": [
        {"nome": "classificador", "duracao_ms": 800, "tokens": 350, "resultado": "multi_step"},
        {"nome": "agent_llm_1", "duracao_ms": 1200, "tokens": 580, "tool_calls": ["search_docs"]},
        {"nome": "search_docs", "duracao_ms": 280, "tokens": 0, "resultado": "3 acórdãos encontrados"},
        {"nome": "agent_llm_2", "duracao_ms": 1100, "tokens": 720, "tool_calls": ["query_db"]},
        {"nome": "query_db", "duracao_ms": 45, "tokens": 0, "resultado": "2 ocorrências encontradas"},
        {"nome": "agent_llm_3", "duracao_ms": 1800, "tokens": 1230, "tool_calls": ["web_search"]},
        {"nome": "web_search", "duracao_ms": 2100, "tokens": 0, "resultado": "3 artigos recentes"},
        {"nome": "agent_llm_4", "duracao_ms": 4875, "tokens": 1943, "tool_calls": []},
    ]
}

print(f"\n📋 TRACE: {trace_simulado['trace_id']}")
print(f"   Input: {trace_simulado['input']}")
print(f"   Total tokens: {trace_simulado['total_tokens']:,}")
print(f"   Custo: ${trace_simulado['custo_estimado_usd']:.5f}")
print(f"   Latência total: {trace_simulado['latencia_total_ms']:,}ms")

print("\n📊 SPANS (passos do agente):")
print(f"{'Span':<20} {'Duração':>10} {'Tokens':>8} {'Observações'}")
print("-" * 70)
for span in trace_simulado["spans"]:
    tc = f"→ {span['tool_calls']}" if span.get("tool_calls") else ""
    res = f"↩ {span.get('resultado', '')[:30]}" if span.get("resultado") else ""
    obs = tc or res
    print(f"  {span['nome']:<18} {span['duracao_ms']:>7}ms {span.get('tokens',0):>8} {obs}")

print("\n🔍 ANÁLISE DO TRACE:")
duracao_llm = sum(s["duracao_ms"] for s in trace_simulado["spans"] if s["nome"].startswith("agent"))
duracao_tools = sum(s["duracao_ms"] for s in trace_simulado["spans"] if not s["nome"].startswith("agent") and s["nome"] != "classificador")
print(f"   Tempo no LLM: {duracao_llm:,}ms ({duracao_llm/trace_simulado['latencia_total_ms']:.0%})")
print(f"   Tempo em tools: {duracao_tools:,}ms ({duracao_tools/trace_simulado['latencia_total_ms']:.0%})")
print(f"   ⚠️  web_search demorou 2100ms — considere cache ou timeout menor")
print(f"   ⚠️  LLM-4 usou {trace_simulado['spans'][-1]['tokens']:,} tokens para a resposta final — possível otimização")

In [ ]:
# Configuração real do LangFuse (executa apenas se keys configuradas)

if os.getenv("LANGFUSE_PUBLIC_KEY") and os.getenv("LANGFUSE_SECRET_KEY"):
    from langfuse import Langfuse
    from langfuse.callback import CallbackHandler
    import datetime
    
    lf = Langfuse(
        public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
        secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
        host="https://cloud.langfuse.com"
    )
    
    # Registra trace manualmente para demonstração
    trace = lf.trace(
        name="aula10-lab3-demo",
        input={"query": "Avaliação Lab 3 Aula 10"},
        metadata={"aula": 10, "lab": 3, "turma": "MBA-2024"}
    )
    
    # Registra geração (chamada ao LLM)
    generation = trace.generation(
        name="classificador_query",
        model="gpt-4o-mini",
        input=[{"role": "user", "content": "Classifique: O que é habeas corpus?"}],
        output="sem_retrieval",
        usage={"input": 150, "output": 30, "total": 180}
    )
    
    # Score de avaliação
    trace.score(
        name="faithfulness",
        value=0.94,
        comment="Avaliação automática RAGAS"
    )
    
    lf.flush()
    print("✅ Trace registrado no LangFuse!")
    print("   Acesse: https://cloud.langfuse.com para visualizar")
else:
    print("ℹ️  Configure LANGFUSE_PUBLIC_KEY e LANGFUSE_SECRET_KEY para enviar traces reais.")
    print("\n📝 COMO CRIAR UMA CONTA LANGFUSE:")
    print("   1. Acesse: https://cloud.langfuse.com")
    print("   2. Crie um projeto: 'rag-juridico-mba'")
    print("   3. Copie as chaves API")
    print("   4. Adicione aos Secrets do Colab")

---
## ✏️ Exercícios

### Exercício 1: Calibração do Threshold
Defina um threshold de aceitação para cada métrica:
- Qual o mínimo de **Faithfulness** aceitável para um parecer jurídico? (Dica: considere responsabilidade legal)
- Qual o máximo de **overhead de custo** tolerável em produção?

### Exercício 2: Análise de Casos Problemáticos
Identifique no Dataset de Avaliação:
1. Qual caso tem a pior trajetória? Por quê?
2. Como você modificaria o prompt do agente para corrigir?

### Exercício 3: Cálculo de ROI do Adaptive RAG
Compare o custo de:
- Usar **sempre Multi-step RAG** (sem classificador) para 1000 queries/dia
- Usar **Adaptive RAG** com a distribuição 30/50/20 (sem/single/multi)

Qual é a economia mensal?